In [ ]:
import sys

print("Python version:")
print(sys.version)

print("\nPython executable:")
print(sys.executable)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print("Všetky potrebné knižnice sú dostupné.")

In [ ]:
from pathlib import Path

                                                                           
CSV_PATH = Path(r"C:\Users\profi\Downloads\orders.csv")

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV sa nenašlo: {CSV_PATH.resolve()}. Upravte CSV_PATH.")

                                                              
orders = pd.read_csv(CSV_PATH, parse_dates=["order_date", "ship_date"])

required = {"order_id", "customer_id", "product_id", "order_date",
            "ship_date", "sales", "quantity", "discount", "profit"}
missing = required - set(orders.columns)
if missing:
    raise ValueError(f"V CSV chýbajú stĺpce: {sorted(missing)}")

print("Počet objednávok:", len(orders))
print("Časový rozsah:", orders["order_date"].min(), "–", orders["order_date"].max())
display(orders.head())

In [ ]:
                        
print("Najstaršia objednávka:", orders["order_date"].min())
print("Najnovšia objednávka:", orders["order_date"].max())

                        
orders_by_year = (
    orders["order_date"]
    .dt.year
    .value_counts()
    .sort_index()
)
display(orders_by_year)

                           
orders_by_month = (
    orders["order_date"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)
display(orders_by_month)

                                
orders_by_month.plot(
    kind="bar",
    figsize=(8, 4),
    title="Počet objednávok podľa mesiaca",
    color="#3577d4"
)
plt.xlabel("Mesiac")
plt.ylabel("Počet objednávok")
plt.xticks(rotation=0)
plt.show()

In [ ]:
CUTOFF = pd.Timestamp("2024-03-01")

historical_orders = orders[
    orders["order_date"] < CUTOFF
].copy()

                                                                    
new_orders = (
    orders[orders["order_date"] >= CUTOFF]
    .sort_values(["order_date", "order_id"])
    .reset_index(drop=True)
)

print("Historical batch:", len(historical_orders))
print("New events:", len(new_orders))
print("Súčet:", len(historical_orders) + len(new_orders), "z", len(orders))

In [ ]:
                                                                  
hist_count = len(historical_orders)
hist_total_sales = historical_orders["sales"].sum()
hist_avg_sales = historical_orders["sales"].mean()
hist_total_profit = historical_orders["profit"].sum()
hist_customers = historical_orders["customer_id"].nunique()
hist_products = historical_orders["product_id"].nunique()

print("Počet objednávok:", hist_count)
print("Celkové tržby (sales):", round(hist_total_sales, 2))
print("Priemerná hodnota objednávky:", round(hist_avg_sales, 2))
print("Celkový zisk (profit):", round(hist_total_profit, 2))
print("Unikátni zákazníci:", hist_customers)
print("Unikátne produkty:", hist_products)

In [ ]:
MICRO_BATCH_SIZE = 5                                                    

                                                        
running_count = 0
running_sum = 0.0
running_profit = 0.0

                                                        
micro_updates = []

for batch_no, start in enumerate(range(0, len(new_orders), MICRO_BATCH_SIZE), start=1):
    end = start + MICRO_BATCH_SIZE
    batch = new_orders.iloc[start:end]                          

                                  
    running_count += len(batch)
    running_sum += batch["sales"].sum()
    running_profit += batch["profit"].sum()
    running_avg = running_sum / running_count

    micro_updates.append({
        "batch": batch_no,
        "rows": f"{start}–{min(end, len(new_orders)) - 1}",
        "batch_size": len(batch),
        "last_order_date": batch["order_date"].max().date(),
        "running_count": running_count,
        "running_sum": round(running_sum, 2),
        "running_avg": round(running_avg, 2),
        "running_profit": round(running_profit, 2),
    })

micro_df = pd.DataFrame(micro_updates)
display(micro_df)

In [ ]:
final_batch_count = len(new_orders)
final_batch_avg = new_orders["sales"].mean()
final_batch_sum = new_orders["sales"].sum()

print("MICRO-BATCH | aktualizácií:", len(micro_df),
      "| počet:", running_count,
      "| súčet:", round(running_sum, 2),
      "| priemer:", round(running_sum / running_count, 2))
print("BATCH       | aktualizácií: 1",
      "| počet:", final_batch_count,
      "| súčet:", round(final_batch_sum, 2),
      "| priemer:", round(final_batch_avg, 2))

assert running_count == final_batch_count
assert abs(running_sum - final_batch_sum) < 1e-6
assert abs(running_sum / running_count - final_batch_avg) < 1e-9
print("\n✅ Výsledky sa zhodujú.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

                                      
axes[0].step(micro_df["batch"], micro_df["running_avg"],
             where="post", color="#159572", linewidth=2, marker="o")
axes[0].axhline(final_batch_avg, color="#3577d4", linestyle="--",
                label=f"Batch priemer = {final_batch_avg:.2f}")
axes[0].set_title("Priebežný priemer (micro-batch)")
axes[0].set_xlabel("Číslo dávky")
axes[0].set_ylabel("Priemerná hodnota objednávky (sales)")
axes[0].set_xticks(micro_df["batch"])
axes[0].legend()
axes[0].grid(alpha=0.2)

                                            
axes[1].step(micro_df["batch"], micro_df["running_count"],
             where="post", color="#c88730", linewidth=2, marker="o")
axes[1].set_title("Priebežný počet objednávok")
axes[1].set_xlabel("Číslo dávky")
axes[1].set_ylabel("running_count")
axes[1].set_xticks(micro_df["batch"])
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "Prístup": ["Batch", "Micro-batch"],
    "Aktualizácií výsledku": [1, len(micro_df)],
    "Prvý výsledok po spracovaní (objednávok)": [len(new_orders), int(micro_df.loc[0, "running_count"])],
    "Konečný priemer": [round(final_batch_avg, 2), round(running_sum / running_count, 2)],
})

display(comparison)